# Synthetic recommendation data

- goal: generate clean fake data for recommender experiments
- output: CSV files
- serving unit: `collaboration`
- event shape: aligned with `interactionEvents`


Files written:

- `projects.csv`
- `collaborations.csv`
- `submissions.csv`
- `users.csv`
- `interaction_events.csv`


In [24]:
import sys
print(sys.executable)


/Users/dw/coding2/MAKE-TUNE3/MAKE-TUNE3/recommendation-system/venv/bin/python


In [25]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
import math
import random
import statistics
import uuid

import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "data" / "synthetic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOW = datetime(2026, 5, 2, 12, 0, 0, tzinfo=timezone.utc)

CONFIG = {
    "num_projects": 14,
    "collaborations_per_project": (2, 5),
    "submissions_per_collaboration": (6, 16),
    "num_users": 240,
    "num_taste_clusters": 6,
    "collaboration_tags_per_item": (1, 3),
    "days_history": 180,
}

TAG_POOL = [
    "house", "techno", "ambient", "drum-and-bass", "garage", "dub", "downtempo",
    "breakbeat", "acid", "synthwave", "electro", "idm", "minimal", "lofi",
    "jungle", "trance", "industrial", "organic", "cinematic", "uk-bass",
]

EVENT_WEIGHTS = {
    "submission_like": 1.0,
    "submission_favorite": 2.0,
    "submission_vote": 3.0,
    "collaboration_like": 0.75,
    "collaboration_favorite": 1.75,
}

CONFIG

{'num_projects': 14,
 'collaborations_per_project': (2, 5),
 'submissions_per_collaboration': (6, 16),
 'num_users': 240,
 'num_taste_clusters': 6,
 'collaboration_tags_per_item': (1, 3),
 'days_history': 180}

## Helpers


In [26]:
def isoformat_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def rand_dt_within_days(days_back: int) -> datetime:
    seconds = random.randint(0, days_back * 24 * 60 * 60)
    return NOW - timedelta(seconds=seconds)


def choose_tags(pool: list[str], count_range: tuple[int, int]) -> list[str]:
    count = random.randint(*count_range)
    return sorted(random.sample(pool, count))


def clamp(value: float, low: float, high: float) -> float:
    return max(low, min(high, value))


def sample_weighted(items: list[tuple[str, float]], k: int) -> list[str]:
    keys = []
    weights = []
    for key, weight in items:
        if weight <= 0:
            continue
        keys.append(key)
        weights.append(weight)
    if not keys:
        return []
    k = min(k, len(keys))
    chosen = set()
    while len(chosen) < k:
        chosen.add(random.choices(keys, weights=weights, k=1)[0])
    return list(chosen)


## Shapes


In [27]:
@dataclass
class Project:
    id: str
    name: str
    createdAt: str


@dataclass
class Collaboration:
    id: str
    projectId: str
    name: str
    status: str
    tags: list[str]
    tagsKey: list[str]
    createdAt: str
    publishedAt: str


@dataclass
class Submission:
    id: str
    projectId: str
    collaborationId: str
    trackPath: str
    createdAt: str


@dataclass
class User:
    id: str
    clusterId: str
    preferredTags: list[str]
    createdAt: str


@dataclass
class InteractionEvent:
    id: str
    userId: str
    projectId: str
    collaborationId: str
    trackPath: str | None
    entityType: str
    eventType: str
    createdAt: str
    tags: list[str]


STATUS_OPTIONS = ["submission", "voting", "completed"]

## 1. Catalog


In [28]:
projects: list[Project] = []
collaborations: list[Collaboration] = []
submissions: list[Submission] = []

for project_index in range(CONFIG["num_projects"]):
    project_id = f"project-{project_index + 1:03d}"
    project_created = rand_dt_within_days(CONFIG["days_history"])
    project = Project(
        id=project_id,
        name=f"Project {project_index + 1}",
        createdAt=isoformat_z(project_created),
    )
    projects.append(project)

    num_collabs = random.randint(*CONFIG["collaborations_per_project"])
    for collab_index in range(num_collabs):
        collab_id = f"collab-{project_index + 1:03d}-{collab_index + 1:02d}"
        collab_created = rand_dt_within_days(CONFIG["days_history"])
        tags = choose_tags(TAG_POOL, CONFIG["collaboration_tags_per_item"])
        published = collab_created + timedelta(hours=random.randint(6, 72))
        status = random.choices(STATUS_OPTIONS, weights=[0.55, 0.20, 0.25], k=1)[0]
        collab = Collaboration(
            id=collab_id,
            projectId=project_id,
            name=f"Collaboration {project_index + 1}-{collab_index + 1}",
            status=status,
            tags=tags,
            tagsKey=tags,
            createdAt=isoformat_z(collab_created),
            publishedAt=isoformat_z(published),
        )
        collaborations.append(collab)

        num_submissions = random.randint(*CONFIG["submissions_per_collaboration"])
        for submission_index in range(num_submissions):
            submission_id = f"sub-{project_index + 1:03d}-{collab_index + 1:02d}-{submission_index + 1:02d}"
            track_path = f"collaborations/{collab_id}/submissions/{submission_id}.mp3"
            submission_created = published + timedelta(hours=random.randint(1, 120))
            submissions.append(
                Submission(
                    id=submission_id,
                    projectId=project_id,
                    collaborationId=collab_id,
                    trackPath=track_path,
                    createdAt=isoformat_z(submission_created),
                )
            )

len(projects), len(collaborations), len(submissions)

(14, 47, 532)

## 2. Users

- users are grouped into taste clusters


In [29]:
cluster_profiles: dict[str, list[str]] = {}
for cluster_index in range(CONFIG["num_taste_clusters"]):
    cluster_profiles[f"cluster-{cluster_index + 1}"] = choose_tags(TAG_POOL, (3, 5))

users: list[User] = []
for user_index in range(CONFIG["num_users"]):
    cluster_id = random.choice(list(cluster_profiles.keys()))
    base_tags = cluster_profiles[cluster_id]
    extra_tags = choose_tags([t for t in TAG_POOL if t not in base_tags], (0, 2))
    preferred_tags = sorted(set(base_tags + extra_tags))
    users.append(
        User(
            id=f"user-{user_index + 1:04d}",
            clusterId=cluster_id,
            preferredTags=preferred_tags,
            createdAt=isoformat_z(rand_dt_within_days(CONFIG["days_history"])),
        )
    )

cluster_profiles

{'cluster-1': ['drum-and-bass', 'idm', 'jungle', 'lofi'],
 'cluster-2': ['acid', 'dub', 'garage', 'industrial', 'lofi'],
 'cluster-3': ['acid', 'jungle', 'lofi', 'organic', 'trance'],
 'cluster-4': ['acid', 'ambient', 'breakbeat', 'jungle'],
 'cluster-5': ['cinematic', 'jungle', 'minimal'],
 'cluster-6': ['dub', 'electro', 'house', 'trance']}

## 3. Events

- biased toward matching tags
- sparse overall
- shaped like the real event log


In [30]:
collab_by_id = {c.id: c for c in collaborations}
project_by_id = {p.id: p for p in projects}
subs_by_collab: dict[str, list[Submission]] = defaultdict(list)
for sub in submissions:
    subs_by_collab[sub.collaborationId].append(sub)

collab_popularity_bias: dict[str, float] = {}
for collab in collaborations:
    collab_popularity_bias[collab.id] = random.uniform(0.75, 1.35)

events: list[InteractionEvent] = []

for user in users:
    candidate_scores: list[tuple[str, float]] = []
    for collab in collaborations:
        tag_overlap = len(set(user.preferredTags) & set(collab.tagsKey))
        base_score = 0.05 + (0.22 * tag_overlap)
        popularity_boost = collab_popularity_bias[collab.id] * 0.18
        freshness_boost = 0.08 if collab.status in {"submission", "voting"} else 0.03
        candidate_scores.append((collab.id, base_score + popularity_boost + freshness_boost))

    interacted_collab_ids = sample_weighted(candidate_scores, random.randint(4, 16))

    for collab_id in interacted_collab_ids:
        collab = collab_by_id[collab_id]
        collab_subs = subs_by_collab[collab_id]
        overlap = len(set(user.preferredTags) & set(collab.tagsKey))
        strength = clamp(0.25 + overlap * 0.2 + random.uniform(-0.05, 0.15), 0.10, 0.95)

        if random.random() < strength * 0.75:
            events.append(
                InteractionEvent(
                    id=str(uuid.uuid4()),
                    userId=user.id,
                    projectId=collab.projectId,
                    collaborationId=collab.id,
                    trackPath=None,
                    entityType="collaboration",
                    eventType="collaboration_like",
                    createdAt=isoformat_z(rand_dt_within_days(CONFIG["days_history"])),
                    tags=collab.tagsKey,
                )
            )

        if random.random() < strength * 0.35:
            events.append(
                InteractionEvent(
                    id=str(uuid.uuid4()),
                    userId=user.id,
                    projectId=collab.projectId,
                    collaborationId=collab.id,
                    trackPath=None,
                    entityType="collaboration",
                    eventType="collaboration_favorite",
                    createdAt=isoformat_z(rand_dt_within_days(CONFIG["days_history"])),
                    tags=collab.tagsKey,
                )
            )

        num_subs_to_touch = min(len(collab_subs), random.randint(1, 5 + overlap))
        chosen_subs = random.sample(collab_subs, num_subs_to_touch)
        favorite_candidates: list[str] = []

        for sub in chosen_subs:
            if random.random() < strength:
                events.append(
                    InteractionEvent(
                        id=str(uuid.uuid4()),
                        userId=user.id,
                        projectId=collab.projectId,
                        collaborationId=collab.id,
                        trackPath=sub.trackPath,
                        entityType="submission",
                        eventType="submission_like",
                        createdAt=isoformat_z(rand_dt_within_days(CONFIG["days_history"])),
                        tags=collab.tagsKey,
                    )
                )
                favorite_candidates.append(sub.trackPath)

        favorite_count = min(len(favorite_candidates), 1 if random.random() < 0.7 else 2)
        for track_path in random.sample(favorite_candidates, k=favorite_count) if favorite_candidates and favorite_count > 0 else []:
            events.append(
                InteractionEvent(
                    id=str(uuid.uuid4()),
                    userId=user.id,
                    projectId=collab.projectId,
                    collaborationId=collab.id,
                    trackPath=track_path,
                    entityType="submission",
                    eventType="submission_favorite",
                    createdAt=isoformat_z(rand_dt_within_days(CONFIG["days_history"])),
                    tags=collab.tagsKey,
                )
            )

        if collab.status in {"voting", "completed"} and favorite_candidates and random.random() < strength * 0.55:
            voted_track = random.choice(favorite_candidates)
            events.append(
                InteractionEvent(
                    id=str(uuid.uuid4()),
                    userId=user.id,
                    projectId=collab.projectId,
                    collaborationId=collab.id,
                    trackPath=voted_track,
                    entityType="submission",
                    eventType="submission_vote",
                    createdAt=isoformat_z(rand_dt_within_days(CONFIG["days_history"])),
                    tags=collab.tagsKey,
                )
            )

# Sort chronologically for easier inspection.
events.sort(key=lambda event: event.createdAt)

len(events)

7246

## 4. Aggregate tables


In [31]:
events_df = pd.DataFrame([asdict(event) for event in events])
collabs_df = pd.DataFrame([asdict(collab) for collab in collaborations])
subs_df = pd.DataFrame([asdict(submission) for submission in submissions])

events_df['eventWeight'] = events_df['eventType'].map(EVENT_WEIGHTS)
events_df['createdAt'] = pd.to_datetime(events_df['createdAt'], utc=True)

events_df.head()

,id,userId,projectId,collaborationId,trackPath,entityType,eventType,createdAt,tags,eventWeight
0,c44410dd-2c6a-4e01-9a7c-38dec74a2a8f,user-0150,project-012,collab-012-02,collaborations/collab-012-02/submissions/sub-0...,submission,submission_like,2025-11-03 12:39:49+00:00,"[breakbeat, dub]",1.0
1,fa849484-39a5-476a-a7a5-909e9e072a14,user-0081,project-013,collab-013-01,collaborations/collab-013-01/submissions/sub-0...,submission,submission_favorite,2025-11-03 12:42:24+00:00,[idm],2.0
2,cc0a7518-c697-4d98-a777-999cd649297e,user-0020,project-009,collab-009-02,collaborations/collab-009-02/submissions/sub-0...,submission,submission_like,2025-11-03 13:18:27+00:00,[acid],1.0
3,98480a28-3064-42a2-bc53-75c6e4814dc0,user-0142,project-004,collab-004-02,collaborations/collab-004-02/submissions/sub-0...,submission,submission_like,2025-11-03 13:34:18+00:00,"[drum-and-bass, electro, synthwave]",1.0
4,caf94d32-3272-431f-9194-5096f60bd4e4,user-0037,project-010,collab-010-02,collaborations/collab-010-02/submissions/sub-0...,submission,submission_favorite,2025-11-03 14:12:37+00:00,[trance],2.0


In [32]:
collaboration_features = (
    events_df
    .groupby(['userId', 'projectId', 'collaborationId', 'eventType'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for event_type in EVENT_WEIGHTS:
    if event_type not in collaboration_features.columns:
        collaboration_features[event_type] = 0

collaboration_weight = (
    events_df
    .groupby(['userId', 'projectId', 'collaborationId'])['eventWeight']
    .sum()
    .reset_index(name='totalEventWeight')
)

collaboration_last_event = (
    events_df
    .groupby(['userId', 'projectId', 'collaborationId'])['createdAt']
    .max()
    .reset_index(name='lastEventAt')
)

collaboration_table = (
    collaboration_features
    .merge(collaboration_weight, on=['userId', 'projectId', 'collaborationId'], how='left')
    .merge(collaboration_last_event, on=['userId', 'projectId', 'collaborationId'], how='left')
    .merge(
        collabs_df[['id', 'status', 'tags', 'tagsKey', 'publishedAt']].rename(columns={'id': 'collaborationId'}),
        on='collaborationId',
        how='left',
    )
)

collaboration_table = collaboration_table.sort_values(
    ['userId', 'lastEventAt', 'totalEventWeight'],
    ascending=[True, False, False],
)

collaboration_table.head()

,userId,projectId,collaborationId,collaboration_favorite,collaboration_like,submission_favorite,submission_like,submission_vote,totalEventWeight,lastEventAt,status,tags,tagsKey,publishedAt
1,user-0001,project-007,collab-007-04,0,0,1,1,0,3.00,2026-04-24 00:22:43+00:00,submission,"[industrial, uk-bass]","[industrial, uk-bass]",2026-01-10T03:32:07Z
0,user-0001,project-003,collab-003-01,0,1,1,3,0,5.75,2026-04-03 07:12:35+00:00,submission,"[breakbeat, dub, jungle]","[breakbeat, dub, jungle]",2026-01-21T00:43:40Z
2,user-0001,project-009,collab-009-02,0,0,1,3,0,5.00,2026-03-28 11:54:51+00:00,voting,[acid],[acid],2026-01-16T19:47:28Z
4,user-0002,project-007,collab-007-01,0,0,2,2,0,6.00,2026-04-16 05:06:49+00:00,completed,[ambient],[ambient],2026-02-04T05:59:49Z
9,user-0002,project-012,collab-012-02,0,0,2,2,0,6.00,2026-04-04 14:53:01+00:00,submission,"[breakbeat, dub]","[breakbeat, dub]",2026-04-14T19:16:02Z


In [33]:
submission_table = (
    events_df[events_df['entityType'] == 'submission']
    .groupby(['userId', 'projectId', 'collaborationId', 'trackPath', 'eventType'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for event_type in ['submission_like', 'submission_favorite', 'submission_vote']:
    if event_type not in submission_table.columns:
        submission_table[event_type] = 0

submission_weight = (
    events_df[events_df['entityType'] == 'submission']
    .groupby(['userId', 'projectId', 'collaborationId', 'trackPath'])['eventWeight']
    .sum()
    .reset_index(name='totalEventWeight')
)

submission_last_event = (
    events_df[events_df['entityType'] == 'submission']
    .groupby(['userId', 'projectId', 'collaborationId', 'trackPath'])['createdAt']
    .max()
    .reset_index(name='lastEventAt')
)

submission_table = (
    submission_table
    .merge(submission_weight, on=['userId', 'projectId', 'collaborationId', 'trackPath'], how='left')
    .merge(submission_last_event, on=['userId', 'projectId', 'collaborationId', 'trackPath'], how='left')
)

submission_table = submission_table.sort_values(
    ['userId', 'lastEventAt', 'totalEventWeight'],
    ascending=[True, False, False],
)

submission_table.head()

,userId,projectId,collaborationId,trackPath,submission_favorite,submission_like,submission_vote,totalEventWeight,lastEventAt
3,user-0001,project-007,collab-007-04,collaborations/collab-007-04/submissions/sub-0...,1,1,0,3.0,2026-04-24 00:22:43+00:00
2,user-0001,project-003,collab-003-01,collaborations/collab-003-01/submissions/sub-0...,1,1,0,3.0,2026-04-03 07:12:35+00:00
4,user-0001,project-009,collab-009-02,collaborations/collab-009-02/submissions/sub-0...,0,1,0,1.0,2026-03-28 11:54:51+00:00
5,user-0001,project-009,collab-009-02,collaborations/collab-009-02/submissions/sub-0...,0,1,0,1.0,2026-03-19 12:13:59+00:00
0,user-0001,project-003,collab-003-01,collaborations/collab-003-01/submissions/sub-0...,0,1,0,1.0,2026-01-14 03:42:11+00:00


In [34]:
collaboration_table.shape, submission_table.shape

((2043, 14), (3703, 9))

## 5. Checks


In [35]:
event_counts = Counter(event.eventType for event in events)
event_counts

Counter({'submission_like': 3703,
         'submission_favorite': 2130,
         'collaboration_like': 798,
         'collaboration_favorite': 381,
         'submission_vote': 234})

In [36]:
user_event_counts = Counter(event.userId for event in events)
summary = {
    'projects': len(projects),
    'collaborations': len(collaborations),
    'submissions': len(submissions),
    'users': len(users),
    'events': len(events),
    'avg_events_per_user': round(statistics.mean(user_event_counts.values()), 2) if user_event_counts else 0,
    'median_events_per_user': statistics.median(user_event_counts.values()) if user_event_counts else 0,
}
summary

{'projects': 14,
 'collaborations': 47,
 'submissions': 532,
 'users': 240,
 'events': 7246,
 'avg_events_per_user': 30.19,
 'median_events_per_user': 30.0}

In [37]:
pd.DataFrame([asdict(event) for event in events]).head()

,id,userId,projectId,collaborationId,trackPath,entityType,eventType,createdAt,tags
0,c44410dd-2c6a-4e01-9a7c-38dec74a2a8f,user-0150,project-012,collab-012-02,collaborations/collab-012-02/submissions/sub-0...,submission,submission_like,2025-11-03T12:39:49Z,"[breakbeat, dub]"
1,fa849484-39a5-476a-a7a5-909e9e072a14,user-0081,project-013,collab-013-01,collaborations/collab-013-01/submissions/sub-0...,submission,submission_favorite,2025-11-03T12:42:24Z,[idm]
2,cc0a7518-c697-4d98-a777-999cd649297e,user-0020,project-009,collab-009-02,collaborations/collab-009-02/submissions/sub-0...,submission,submission_like,2025-11-03T13:18:27Z,[acid]
3,98480a28-3064-42a2-bc53-75c6e4814dc0,user-0142,project-004,collab-004-02,collaborations/collab-004-02/submissions/sub-0...,submission,submission_like,2025-11-03T13:34:18Z,"[drum-and-bass, electro, synthwave]"
4,caf94d32-3272-431f-9194-5096f60bd4e4,user-0037,project-010,collab-010-02,collaborations/collab-010-02/submissions/sub-0...,submission,submission_favorite,2025-11-03T14:12:37Z,[trance]


## 6. Export


In [38]:
project_rows = [asdict(project) for project in projects]
collaboration_rows = [asdict(collab) for collab in collaborations]
submission_rows = [asdict(submission) for submission in submissions]
user_rows = [asdict(user) for user in users]
event_rows = [asdict(event) for event in events]

pd.DataFrame(project_rows).to_csv(OUTPUT_DIR / 'projects.csv', index=False)
pd.DataFrame(collaboration_rows).to_csv(OUTPUT_DIR / 'collaborations.csv', index=False)
pd.DataFrame(submission_rows).to_csv(OUTPUT_DIR / 'submissions.csv', index=False)
pd.DataFrame(user_rows).to_csv(OUTPUT_DIR / 'users.csv', index=False)
pd.DataFrame(event_rows).to_csv(OUTPUT_DIR / 'interaction_events.csv', index=False)
collaboration_table.to_csv(OUTPUT_DIR / 'collaboration_training_table.csv', index=False)
submission_table.to_csv(OUTPUT_DIR / 'submission_training_table.csv', index=False)

sorted(str(path.relative_to(ROOT)) for path in OUTPUT_DIR.iterdir())

['data/synthetic/collaboration_training_table.csv',
 'data/synthetic/collaborations.csv',
 'data/synthetic/interaction_events.csv',
 'data/synthetic/projects.csv',
 'data/synthetic/submission_training_table.csv',
 'data/synthetic/submissions.csv',
 'data/synthetic/users.csv']

## Next

- add cold users explicitly
- add holdout splits
- compare the two model directions
